01 - Build the PNG mosaic corpus
#
# Converts the mounted DICOM series into one lossless PNG mosaic per selected
# series (~8-9 GB for the full training set, inside the 20 GB output cap).
# This kernel's output mounts straight into the training notebook as a kernel
# source, so there is no dataset to build by hand.
#
# It is resumable: if the 12 h session limit hits, re-run and it skips what is
# already written.

In [ ]:
import sys, subprocess
from pathlib import Path

CODE = next(
    (p for pat in ("rsna-knee-code", "*/rsna-knee-code", "*/*/rsna-knee-code")
     for p in sorted(Path("/kaggle/input").glob(pat)) if (p / "src").is_dir()),
    None,
)
assert CODE is not None, "rsna-knee-code dataset is not attached to this notebook"
sys.path.insert(0, str(CODE / "src"))
sys.path.insert(0, str(CODE))
print("CODE", CODE)

# timm ships in the Kaggle image; the install is a fallback for images that
# lack it, and cannot run in the offline submission notebook.
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm

import rsna_knee.config as cfg
print("RAW ", cfg.RAW, cfg.RAW.exists())
print("PREP", cfg.PREP, cfg.PREP.exists())

In [ ]:
!cd {CODE} && python scripts/build_weak_labels.py

In [ ]:
# Full corpus. Raise --workers to the number of available cores.
!cd {CODE} && python scripts/preprocess_series.py --split train --workers 4

In [ ]:
!cd {CODE} && python scripts/preprocess_series.py --split test --workers 4

In [ ]:
from pathlib import Path
prep = Path("/kaggle/working/prep")
pngs = list(prep.rglob("*.png"))
size = sum(p.stat().st_size for p in pngs)
print(f"{len(pngs)} mosaics, {size/1e9:.2f} GB")